# 03 · NHTSA Recalls API (vehicles, tires, car seats, equipment)

**Agency:** National Highway Traffic Safety Administration  ·  **Auth:** none  ·  **Updated:** daily
**Endpoints**
- `https://api.nhtsa.gov/recalls/recallsByVehicle?make=&model=&modelYear=`
- `https://api.nhtsa.gov/recalls/campaignNumber?campaignNumber=`
- Bulk: `FLAT_RCL_POST_2010.zip` (+ data dictionary `RCL.txt`) — best for the full backfill, and the place where **child car seats** and other equipment live.

> Saved samples: 20 recalls across 2018 Honda CR-V, 2021 Ford F-150, 2021 Jeep Wrangler, 2019 Toyota RAV4 — including "park it" / "park outside" flags.

In [1]:
import sys, json, re
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so we can import recall_explore from the repo root
import pandas as pd
from recall_explore import fetch, load_samples, find_ids, coverage, normalize_model
pd.set_option("display.max_colwidth", 80)

LIVE = False   # True = call the real API (needs internet). False = use the 20 saved samples.

In [2]:
if LIVE:
    recalls = []
    for make, model, yr in [("honda","cr-v",2018), ("ford","f-150",2021), ("jeep","wrangler",2021), ("toyota","rav4",2019)]:
        res = fetch("nhtsa_vehicle", make=make, model=model, modelYear=yr)
        print(make, model, yr, "→", res["Count"], "recalls")
        recalls += res["results"]
else:
    recalls = load_samples("nhtsa")
print(len(recalls), "records loaded")
# Seen on 2026-10-20: CR-V 2018 → 5, F-150 2021 → 29, Wrangler 2021 → 15, RAV4 2019 → 7

20 records loaded


## 1 · Anatomy
The cleanest schema of the three: flat, consistent keys, structured make/model/year.

In [3]:
r = recalls[0]
for k, v in r.items():
    print(f"{k:<20}", json.dumps(v)[:110])

Manufacturer         "Honda (American Honda Motor Co.)"
NHTSACampaignNumber  "18V663000"
parkIt               false
parkOutSide          false
overTheAirUpdate     false
ReportReceivedDate   "27/09/2018"
Component            "STEERING:ELECTRIC POWER ASSIST SYSTEM"
Summary              "Honda (American Honda Motor Co.) is recalling certain 2017-2018 Honda Civic and CR-V vehicles.  The magnet th
Consequence          "The unintended steering input reduces vehicle maneuverability and increases the risk of a crash."
Remedy               "Honda will notify owners, and dealers will replace the steering gearbox assembly, free of charge.  The recall
Notes                "Owners may also contact the National Highway Traffic Safety Administration Vehicle Safety Hotline at 1-888-32
ModelYear            "2018"
Make                 "HONDA"
Model                "CR-V"


In [4]:
coverage(recalls, ["NHTSACampaignNumber", "Manufacturer", "Make", "Model", "ModelYear",
                   "Component", "Summary", "Consequence", "Remedy", "ReportReceivedDate"])

,filled,of,pct
field,,,
NHTSACampaignNumber,20,20,100
Manufacturer,20,20,100
Make,20,20,100
Model,20,20,100
ModelYear,20,20,100
Component,20,20,100
Summary,20,20,100
Consequence,20,20,100
Remedy,20,20,100


## 2 · Severity flags
`parkIt` = don't drive it. `parkOutSide` = fire risk, park away from buildings. `overTheAirUpdate` = fixed by software update.

In [5]:
df = pd.DataFrame(recalls)
df[["parkIt", "parkOutSide", "overTheAirUpdate"]].sum()

parkIt              1
parkOutSide         3
overTheAirUpdate    0
dtype: int64

In [6]:
df[df.parkIt | df.parkOutSide][["NHTSACampaignNumber", "Make", "Model", "Component", "parkIt", "parkOutSide"]]

,NHTSACampaignNumber,Make,Model,Component,parkIt,parkOutSide
9,21V407000,FORD,F-150,STEERING:GEAR BOX (OTHER THAN RACK AND PINION),True,False
15,23V787000,JEEP,WRANGLER,ELECTRICAL SYSTEM:PROPULSION SYSTEM:TRACTION BATTERY,False,True
17,25V741000,JEEP,WRANGLER,ELECTRICAL SYSTEM:PROPULSION SYSTEM:TRACTION BATTERY,False,True
19,26V363000,JEEP,WRANGLER,STEERING:HYDRAULIC POWER ASSIST:PUMP,False,True


## 3 · Gotcha: the date format is DD/MM/YYYY
`ReportReceivedDate = "25/03/2021"` — day first. Parsing it as US month-first would silently break or swap dates.

In [7]:
df["received"] = pd.to_datetime(df.ReportReceivedDate, format="%d/%m/%Y")
df[["NHTSACampaignNumber", "ReportReceivedDate", "received"]].head()

,NHTSACampaignNumber,ReportReceivedDate,received
0,18V663000,27/09/2018,2018-09-27
1,19V503000,28/06/2019,2019-06-28
2,20V064000,06/02/2020,2020-02-06
3,20V286000,20/05/2020,2020-05-20
4,20V373000,24/06/2020,2020-06-24


## 4 · Gotcha: "your model" ≠ "your car"
A make/model/year query returns every recall that **includes** that model — but the `Summary` usually says *"certain"* vehicles, and many cover a sub-type (hybrid, manual transmission, a production window).

In [8]:
qualifiers = r"(plug-in|phev|4xe|hybrid|manual transmission|equipped with|built from|manufactured)"
df["narrowed_by"] = df.Summary.str.extract(qualifiers, flags=re.I)[0]
print(f"{df.Summary.str.contains('certain', case=False).sum()}/{len(df)} summaries say 'certain' vehicles")
df[df.narrowed_by.notna()][["NHTSACampaignNumber", "Model", "narrowed_by", "Summary"]].head(8)

20/20 summaries say 'certain' vehicles


,NHTSACampaignNumber,Model,narrowed_by,Summary
1,19V503000,RAV4,equipped with,"Southeast Toyota Distributors, LLC (SET) is recalling certain 2017-2019 Toyo..."
2,20V064000,RAV4,Hybrid,Toyota Motor Engineering & Manufacturing (Toyota) is recalling certain 2020 ...
3,20V286000,RAV4,Hybrid,Toyota Motor Engineering & Manufacturing (Toyota) is recalling certain 2019-...
4,20V373000,RAV4,Hybrid,Toyota Motor Engineering & Manufacturing (Toyota) is recalling certain 2019-...
6,21V028000,WRANGLER,equipped with,"Chrysler (FCA US, LLC) is recalling certain 2018-2021 Jeep Wrangler and 2020..."
8,21V215000,CR-V,Hybrid,"Honda (American Honda Motor Co.) is recalling certain 2019-2020 Acura MDX, M..."
14,23V158000,CR-V,Hybrid,"Honda (American Honda Motor Co.) is recalling certain 2017-2020 CR-V, 2018-2..."
15,23V787000,WRANGLER,Plug-In,"Chrysler (FCA US, LLC) is recalling certain 2021-2023 Jeep Wrangler Plug-In ..."


⚠️ Look closely at that table: the qualifier regex is itself misleading. `21V215000` is flagged *Hybrid* only because the Summary lists "MDX Sport Hybrid" — a different model. Deciding *which* qualifier applies to *which* model is, again, a job for LLM extraction, not regex.

Example: `23V787000` comes back for *any* 2021 Wrangler, but it only applies to the **Plug-In Hybrid (4xe)**. If a user owns a gas Wrangler, that would be a false alarm.
Only the VIN-level check on NHTSA.gov knows whether a *specific* car is affected **and still unrepaired**. So:
- Our VIN feature = decode VIN → make/model/year/trim → show recalls **for the model**, clearly labelled, + deep-link to NHTSA's official VIN check.
- One campaign shows up under **many** models (`21V215000` fuel pump covers ~20 Honda/Acura models) → store campaign once, link many vehicles.

In [9]:
df.groupby("Component").size().sort_values(ascending=False).head(10)

Component
FUEL SYSTEM, GASOLINE:DELIVERY:FUEL PUMP                   3
ELECTRICAL SYSTEM:PROPULSION SYSTEM:TRACTION BATTERY       2
STEERING:ELECTRIC POWER ASSIST SYSTEM                      2
AIR BAGS:SENSOR:SIDE IMPACT                                1
BACK OVER PREVENTION:DISPLAY FUNCTION                      1
ENGINE AND ENGINE COOLING                                  1
EQUIPMENT:OTHER:LABELS                                     1
AIR BAGS:SENSOR:OCCUPANT CLASSIFICATION:FRONT PASSENGER    1
POWER TRAIN:CLUTCH ASSEMBLY                                1
POWER TRAIN:DRIVELINE:DRIVESHAFT                           1
dtype: int64

Note `EQUIPMENT:OTHER:LABELS` — a sticker/label recall. Those are real but low-stakes; rank them below airbags and brakes.

## ✅ Takeaways for Recall Radar
1. NHTSA is the **easiest to ingest** (clean, structured) — great first data source for the week-4 MVP.
2. Parse `ReportReceivedDate` as **DD/MM/YYYY**.
3. Use `parkIt` / `parkOutSide` for top-priority alerts, `Component` for severity tiers.
4. Model-level matching over-matches (trims, hybrids). Be honest in the UI and link to NHTSA's VIN check.
5. Car seats come from NHTSA's **bulk file**, not the vehicle endpoint → add a backfill task for it.